# 2.1 MCP (Model Context Protocol)

MCP is a standard way to expose **tools**, **resources**, and **prompts** from a server. LangChain's `MultiServerMCPClient` loads those into an agent so you do not re-implement each integration.

This notebook:

1. Connects to a **local** MCP server (`resources/2.1_mcp_server.py`) over stdio.
2. Pulls tools, resources, and a prompt, then builds an agent.
3. Connects to a **packaged** MCP server (`mcp_server_time`) for the current time.

Run cells from the `notebooks/module-2` directory so the local `resources/` path resolves. The time server needs `uv`.

## Setup

Load API keys, then apply a Windows-only asyncio fix so Jupyter can spawn MCP subprocesses.

In [4]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

`transport: "stdio"` starts the server as a child process. `command` + `args` is how you launch it (`python resources/2.1_mcp_server.py`).

That server (FastMCP) exposes:

- a **tool** (`search_web` via Tavily)
- a **resource** (LangChain MCP adapters README from GitHub)
- a **prompt** (system instructions for LangChain Q&A)

In [6]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Key "local_server" is the server name used in get_resources / get_prompt
client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                # sys.executable so the server runs in this kernel's environment
                "command": sys.executable,
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [7]:
# get tools — LangChain Tool objects ready for create_agent
tools = await client.get_tools()

# get resources — named server must match the dict key above
resources = await client.get_resources("local_server")

# get prompts — "prompt" is the template name on the MCP server
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

Wire MCP tools and the MCP prompt into `create_agent`. Use `ainvoke` because the MCP client is async.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=prompt
)

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

# Async invoke matches the async MCP client
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [ ]:
from pprint import pprint

# Expect tool/resource calls in the message list, then a final answer
pprint(response)

## Packaged MCP server (time)

Still stdio, but the process is `uv run python -m mcp_server_time`. That is a published MCP package, not a local file.

`--local-timezone=America/New_York` is passed through to the server.

In [ ]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [ ]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)